In [ ]:
import os
if not os.path.exists("methane-eos-regression"):
    !git clone https://github.com/somya-assery/methane-eos-regression.git
os.chdir("methane-eos-regression")
print(os.listdir("data/raw"))

In [ ]:
import pandas as pd

R = 8.314462618        # J/(mol K)
M = 0.01604246         # kg/mol, methane molar mass
TC = 190.564           # K, critical temperature
RHOC = 162.66 / M      # mol/m3, critical density
RAW = "data/raw"

cri = pd.read_csv(f"{RAW}/cristancho_2010.csv")
dou = pd.read_csv(f"{RAW}/douslin_1964.csv")
goo = pd.read_csv(f"{RAW}/goodwin_prydz_1972.csv")
ven = pd.read_csv(f"{RAW}/vennix_1965.csv")

# convert to SI: T_K, P_Pa, rho_mol_m3
cri_si = pd.DataFrame({"T_K": cri.T_K, "P_Pa": cri.P_MPa*1e6, "rho_mol_m3": cri.rho_kg_m3/M})
dou_si = pd.DataFrame({"T_K": dou.T_K, "P_Pa": dou.P_bar*1e5, "rho_mol_m3": dou.rho_mol_L*1e3})
goo_si = pd.DataFrame({"T_K": goo.T_K, "P_Pa": goo.P_bar*1e5, "rho_mol_m3": goo.rho_mol_L*1e3})
ven_si = pd.DataFrame({"T_K": ven.T_K, "P_Pa": ven.P_bar*1e5, "rho_mol_m3": ven.rho_mol_L*1e3})

cri_si["source"], cri_si["region"] = "Cristancho 2010", "gas/supercritical"
dou_si["source"], dou_si["region"] = "Douslin 1964", "gas/supercritical"
goo_si["source"], goo_si["region"] = "Goodwin & Prydz 1972", "cold liquid"
ven_si["source"], ven_si["region"] = "Vennix 1965", "liquid/dense fluid"

# rows kept in the raw files but excluded from fitting
goo_si["exclude_reason"] = ""
goo_si.loc[(goo.id == 1) & (goo.T_K > 150), "exclude_reason"] = "critical point row"
goo_si.loc[(goo.id == 1) & (goo.T_K < 150), "exclude_reason"] = "triple point row"
ven_si["exclude_reason"] = ""
ven_si.loc[(ven.rho_mol_L == 18.411) & (ven.T_K == 272.927), "exclude_reason"] = "suspect: outlier in isochore"

df = pd.concat([cri_si, dou_si, goo_si, ven_si], ignore_index=True)
df["exclude_reason"] = df["exclude_reason"].fillna("")
df["Z"] = df.P_Pa / (df.rho_mol_m3 * R * df.T_K)
df["delta"] = df.rho_mol_m3 / RHOC      # reduced density
df["tau"] = TC / df.T_K                 # inverse reduced temperature

usable = df[df.exclude_reason == ""].copy()
print(usable.groupby("source").size())
print("usable:", len(usable), "| excluded:", (df.exclude_reason != "").sum())
df.head()

In [ ]:
import numpy as np, pandas as pd, warnings
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV, LassoCV, BayesianRidge, ARDRegression, LinearRegression
from sklearn.model_selection import GroupKFold
warnings.filterwarnings("ignore")

data = usable.reset_index(drop=True)
y = (data.Z - 1.0).values            # target: Z - 1
sources = data.source.values

def term_bank(d, tau, dmax=6, tmax=5):
    cols = [d**k * tau**t for k in range(1, dmax+1) for t in range(0, tmax+1)]
    names = [f"d{k}_t{t}" for k in range(1, dmax+1) for t in range(0, tmax+1)]
    return np.column_stack(cols), names

def virial_feats(d, tau):            # few-term virial baseline
    return np.column_stack([d, d*tau, d*tau**2, d**2, d**2*tau])

Xbank, names = term_bank(data.delta.values, data.tau.values)
Xvir = virial_feats(data.delta.values, data.tau.values)

def fit_predict(name, tr, te):
    if name == "ideal gas":
        return np.zeros(te.sum())
    if name == "virial OLS":
        return LinearRegression(fit_intercept=False).fit(Xvir[tr], y[tr]).predict(Xvir[te])
    sc = StandardScaler().fit(Xbank[tr])                 # scaler fitted on training sources only
    Xtr, Xte = sc.transform(Xbank[tr]), sc.transform(Xbank[te])
    # tuning uses held-out sources INSIDE the training set, never the test source
    inner = list(GroupKFold(n_splits=len(np.unique(sources[tr]))).split(Xtr, y[tr], sources[tr]))
    if name == "ridge":
        m = RidgeCV(alphas=np.logspace(-6, 3, 30), cv=inner).fit(Xtr, y[tr])
    elif name == "lasso":
        m = LassoCV(alphas=np.logspace(-7, 0, 30), cv=inner, max_iter=200000).fit(Xtr, y[tr])
    elif name == "bayesian ridge":
        m = BayesianRidge().fit(Xtr, y[tr])
    elif name == "ARD":
        m = ARDRegression(max_iter=300).fit(Xtr, y[tr])
    return m.predict(Xte)

rows = []
for held in np.unique(sources):                          # leave-one-source-out
    te = sources == held; tr = ~te
    for name in ["ideal gas", "virial OLS", "ridge", "lasso", "bayesian ridge", "ARD"]:
        Zp = 1 + fit_predict(name, tr, te); Zt = data.Z.values[te]
        rows.append(dict(held_out=held, model=name, n=te.sum(),
                         MAE_Z=np.mean(np.abs(Zp - Zt)),
                         med_pct=np.median(100*np.abs(Zp - Zt)/Zt)))
res = pd.DataFrame(rows)
print(res.pivot(index="model", columns="held_out", values="MAE_Z").round(4).to_string())
print(res.pivot(index="model", columns="held_out", values="med_pct").round(2).to_string())

In [ ]:
from sklearn.linear_model import Lasso
alphas = np.logspace(-7, -1, 14)
rows = []
for a in alphas:
    sc = StandardScaler().fit(Xbank)
    n_terms = int((Lasso(alpha=a, max_iter=200000).fit(sc.transform(Xbank), y).coef_ != 0).sum())
    row = {"alpha": f"{a:.1e}", "terms_all_data": n_terms}
    for held in np.unique(sources):
        te = sources == held; tr = ~te
        sc2 = StandardScaler().fit(Xbank[tr])
        m = Lasso(alpha=a, max_iter=200000).fit(sc2.transform(Xbank[tr]), y[tr])
        p = m.predict(sc2.transform(Xbank[te]))
        row[held] = np.median(100*np.abs(p - y[te])/data.Z.values[te])   # median % error in Z
    rows.append(row)
path = pd.DataFrame(rows)
print(path.round(2).to_string(index=False))

In [ ]:
out = []
for held in np.unique(sources):
    te = sources == held; tr = ~te
    sc2 = StandardScaler().fit(Xbank[tr])
    m = BayesianRidge().fit(sc2.transform(Xbank[tr]), y[tr])
    mu, sd = m.predict(sc2.transform(Xbank[te]), return_std=True)
    inside = np.abs(y[te] - mu) <= 1.96 * sd
    out.append(dict(held_out=held, coverage_95=inside.mean(), mean_sd=sd.mean(), mean_abs_err=np.abs(y[te]-mu).mean()))
print(pd.DataFrame(out).round(4).to_string(index=False))

In [ ]:
from sklearn.linear_model import LassoCV, RidgeCV, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold

# (1) the compact model: fit on all usable data and find optimal terms with CV
sc = StandardScaler().fit(Xbank)
inner_all = list(GroupKFold(n_splits=len(np.unique(sources))).split(Xbank, y, sources))
final = LassoCV(alphas=np.logspace(-7, 0, 30), cv=inner_all, max_iter=200000).fit(sc.transform(Xbank), y)
coef = pd.Series(final.coef_, index=names)
kept = coef[coef != 0]
print(f"compact lasso CV: {len(kept)} of {len(names)} terms kept (alpha = {final.alpha_:.2e})")
print(kept.round(4).sort_values(key=np.abs, ascending=False).to_string())

# (2) models used in the extra splits
def fit_pred(name, tr, te):
    if name == "ideal gas": return np.zeros(te.sum())
    if name == "virial OLS": return LinearRegression(fit_intercept=False).fit(Xvir[tr], y[tr]).predict(Xvir[te])
    s = StandardScaler().fit(Xbank[tr]); A, B = s.transform(Xbank[tr]), s.transform(Xbank[te])
    inner = list(GroupKFold(n_splits=len(np.unique(sources[tr]))).split(A, y[tr], sources[tr]))
    if name == "ridge": return RidgeCV(alphas=np.logspace(-6, 3, 30), cv=inner).fit(A, y[tr]).predict(B)
    if name == "lasso CV": return LassoCV(alphas=np.logspace(-7, 0, 30), cv=inner, max_iter=200000).fit(A, y[tr]).predict(B)

models = ["ideal gas", "virial OLS", "ridge", "lasso CV"]

# groups: whole isotherms (Cristancho, Douslin) and whole isochores (Goodwin, Vennix)
g = data.source.astype(str).copy()
is_iso = data.source.isin(["Cristancho 2010", "Douslin 1964"])
g[is_iso] = data.source[is_iso] + "|T" + data.T_K[is_iso].round(0).astype(int).astype(str)
for s in ["Goodwin & Prydz 1972", "Vennix 1965"]:
    m = (data.source == s).values
    blk = (data.loc[m, "rho_mol_m3"].diff().abs() > 200).cumsum()   # new block when density jumps > 0.2 mol/L
    g[m] = s + "|b" + blk.astype(str)
groups = g.values
print("held-out groups:", pd.Series(groups).nunique())

def evaluate(test_masks):
    rows = []
    for te in test_masks:
        tr = ~te
        for mname in models:
            Zp = 1 + fit_pred(mname, tr, te); Zt = data.Z.values[te]
            for s in np.unique(sources[te]):
                k = sources[te] == s
                rows.append(dict(source=s, model=mname, pct=100*np.abs(Zp[k]-Zt[k])/Zt[k]))
    out = []
    for (s, mname), d in pd.DataFrame(rows).groupby(["source", "model"]):
        v = np.concatenate(d.pct.values)
        out.append(dict(source=s, model=mname, n=len(v), median_pct=np.median(v), max_pct=v.max()))
    return pd.DataFrame(out)

# split A: leave one whole isotherm/isochore out
resA = evaluate([groups == gg for gg in np.unique(groups)])
print("\nHeld-out isotherms/isochores, median % error in Z:")
print(resA.pivot(index="model", columns="source", values="median_pct").round(2).to_string())
print("worst single error (%):")
print(resA.pivot(index="model", columns="source", values="max_pct").round(1).to_string())

# split B: train on P < 40 MPa, test on P >= 40 MPa
hp = data.P_Pa.values >= 40e6
resB = evaluate([hp])
print(f"\nHigh-pressure test (P >= 40 MPa), {hp.sum()} rows, median % error in Z:")
print(resB.pivot(index="model", columns="source", values="median_pct").round(2).to_string())
print("worst single error (%):")
print(resB.pivot(index="model", columns="source", values="max_pct").round(1).to_string())


In [ ]:
from sklearn.linear_model import LassoCV, RidgeCV, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold

# near-critical rule, fixed before looking at these results (write the same rule in your README)
NEAR_T, NEAR_DELTA = 10.0, (0.6, 1.6)      # within 10 K of Tc and reduced density 0.6 to 1.6
near = (((data.T_K - TC).abs() < NEAR_T) & data.delta.between(*NEAR_DELTA)).values
print("near-critical rows by source:"); print(data[near].groupby("source").size().to_string())
print("total:", near.sum(), "of", len(data))

def fit_pred(name, tr, te):
    if name == "ideal gas": return np.zeros(te.sum())
    if name == "virial OLS": return LinearRegression(fit_intercept=False).fit(Xvir[tr], y[tr]).predict(Xvir[te])
    s = StandardScaler().fit(Xbank[tr]); A, B = s.transform(Xbank[tr]), s.transform(Xbank[te])
    inner = list(GroupKFold(n_splits=len(np.unique(sources[tr]))).split(A, y[tr], sources[tr]))
    if name == "ridge": return RidgeCV(alphas=np.logspace(-6, 3, 30), cv=inner).fit(A, y[tr]).predict(B)
    if name == "lasso CV": return LassoCV(alphas=np.logspace(-7, 0, 30), cv=inner, max_iter=200000).fit(A, y[tr]).predict(B)

models = ["ideal gas", "virial OLS", "ridge", "lasso CV"]
rows = []
for held in np.unique(sources):
    tr = (sources != held) & ~near               # train WITHOUT near-critical rows
    for region, mask in [("away from critical point", ~near), ("near critical point", near)]:
        te = (sources == held) & mask
        if te.sum() == 0: continue
        for m in models:
            Zp = 1 + fit_pred(m, tr, te); Zt = data.Z.values[te]
            pct = 100*np.abs(Zp - Zt)/Zt
            rows.append(dict(held_out=held, region=region, model=m, n=te.sum(),
                             median_pct=np.median(pct), max_pct=pct.max()))
res = pd.DataFrame(rows)
for reg in ["away from critical point", "near critical point"]:
    d = res[res.region == reg]
    print(f"\n{reg}: median % error in Z")
    print(d.pivot(index="model", columns="held_out", values="median_pct").round(2).to_string())
    print("rows per source:", d[d.model == "ideal gas"].set_index("held_out").n.to_dict())
    print("worst single error (%):")
    print(d.pivot(index="model", columns="held_out", values="max_pct").round(1).to_string())
